In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
 pip install -q tensorflow-model-optimization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 5.5 MB/s eta 0:00:00


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score
from tensorflow_model_optimization.python.core.keras.compat import keras
import tempfile
import zipfile
import os

In [4]:
batch_size = 64
img_height = 224
img_width = 224

data_dir = "/content/drive/MyDrive/TinyML/Datasets/Tomatoes Diseases/"

In [17]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir + "Train",
    validation_split=0.2,
    subset="training",
    seed=440,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True,
)

Found 12860 files belonging to 10 classes.
Using 10288 files for training.


In [5]:
test_ds = keras.utils.image_dataset_from_directory(
    data_dir + "Test",
    validation_split=0.2,
    subset="training",
    seed=48,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True,
)

Found 3198 files belonging to 10 classes.
Using 2559 files for training.


In [19]:
class_names = train_ds.class_names
class_names

['Bacterial_spot',
 'Early_blight',
 'Late_blight',
 'Leaf_Mold',
 'Septoria_leaf_spot',
 'Spider_mite_Two_spotted_sm',
 'Target_Spot',
 'Yellow_Leaf__Curl_Virus',
 'healthy',
 'mosaic_virus']

In [20]:
test_ds = test_ds.take(16)
train_ds = train_ds.take(125)

In [30]:
AUTOTUNE = tf.data.AUTOTUNE

test_nds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)
train_nds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [ ]:
mobilenet = "drive/MyDrive/TinyML/SaveModels/MobileNet.h5"

In [38]:
def get_gzipped_model_size(file, x):
  # Returns size of gzipped model, in bytes.
  zipped_file = "drive/MyDrive/TinyML/SaveModels/" + x
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(file)

  return os.path.getsize(zipped_file) / 1024

In [ ]:
print("MobileNet size: ", get_gzipped_model_size(mobilenet, "MobileNet.zip"), ' KB')

MobileNet size:  8271.1025390625  KB


In [ ]:
pruned_mobilenet = "drive/MyDrive/TinyML/SaveModels/pruned_MobileNet.h5"

In [ ]:
print("Pruned MobileNet size: ", get_gzipped_model_size(pruned_mobilenet, "pruned_MobileNet.zip"), ' KB')

Pruned MobileNet size:  3755.2353515625  KB


In [22]:
quant_mobilenet = keras.models.load_model("drive/MyDrive/TinyML/SaveModels/pruned_MobileNet.h5")
quant_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
quant_mobilenet.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d_1  (None, 1280)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dropout_1 (Dropout)         (None, 1280)              0         
                                                                 
 dense_1 (Dense)             (None, 10)                12810     
                                                                 
Total params: 2270794 (8.66 MB)
Trainable params: 2236682 (

In [ ]:
quant_mobilenet.evaluate(test_nds)

16/16 [==============================] - 3s 88ms/step - loss: 0.1610 - accuracy: 0.9424


[0.16101962327957153, 0.9423828125]

In [35]:
def representative_data_gen():
  for image, label in train_nds.take(5):
    for img  in image:
      image = np.array(img, dtype=np.float32, ndmin=4)
      yield [image]

In [44]:
converter = tf.lite.TFLiteConverter.from_keras_model(quant_mobilenet)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_data_gen

#converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
#converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.SELECT_TF_OPS]
quant_mobilenet_tflite = converter.convert()

/usr/local/lib/python3.10/dist-packages/tensorflow/lite/python/convert.py:983: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


ConverterError: Could not translate MLIR to FlatBuffer./usr/lib/python3.10/runpy.py:196:1: error: 'tf.FusedBatchNormV3' op is neither a custom op nor a flex op
    return _run_code(code, main_globals, None,
^
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall"]): called from
/usr/lib/python3.10/runpy.py:196:1: note: Error code: ERROR_NEEDS_FLEX_OPS
    return _run_code(code, main_globals, None,
^
/usr/lib/python3.10/runpy.py:196:1: error: 'tf.FusedBatchNormV3' op is neither a custom op nor a flex op
    return _run_code(code, main_globals, None,
^
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall"]): called from
/usr/lib/python3.10/runpy.py:196:1: note: Error code: ERROR_NEEDS_FLEX_OPS
    return _run_code(code, main_globals, None,
^
/usr/lib/python3.10/runpy.py:196:1: error: 'tf.FusedBatchNormV3' op is neither a custom op nor a flex op
    return _run_code(code, main_globals, None,
^
<unknown>:0: note: loc(fused["StatefulP

In [39]:
tflite_name = "drive/MyDrive/TinyML/SaveModels/ptq_MobileNet.tflite"

with open(tflite_name, "wb") as f:
  f.write(quant_mobilenet_tflite)


print("PTQ TFLite MobileNet size:", get_gzipped_model_size(tflite_name, "ptq_MobileNet.zip"), ' KB')

PTQ TFLite MobileNet size: 1469.744140625  KB


In [40]:
def eval_model(test_ds):
  interpreter = tf.lite.Interpreter(tflite_name)
  interpreter.allocate_tensors()

  input_index = interpreter.get_input_details()[0]["index"]
  output_index = interpreter.get_output_details()[0]["index"]

  prediction_digits = []
  vrai = []
  i = 0
  nbre = 0
  num_correct = 0

  for images, labels in test_ds:

    for img, lab in zip(images, labels):
      if i%100 == 0:
        print(f"Evaluated on {i} results so far.")
      i += 1

      img = np.expand_dims(img, axis=0).astype(np.float32)
      #img = (np.expand_dims(img, axis=0) * 255).astype(np.int8)
      interpreter.set_tensor(input_index, img)

      interpreter.invoke()

      output = interpreter.get_tensor(output_index)
      digit = np.argmax(output)
      prediction_digits.append(digit)


      vrai.append(lab)

      if digit == lab:
        num_correct += 1

  prediction_digits = np.array(prediction_digits)

  return num_correct, prediction_digits, vrai

In [41]:
n, predict, lab = eval_model(test_nds)

Evaluated on 0 results so far.
Evaluated on 100 results so far.
Evaluated on 200 results so far.
Evaluated on 300 results so far.
Evaluated on 400 results so far.
Evaluated on 500 results so far.
Evaluated on 600 results so far.
Evaluated on 700 results so far.
Evaluated on 800 results so far.
Evaluated on 900 results so far.
Evaluated on 1000 results so far.


In [42]:
print(f"{n}/{1024}\n")

150/1024



In [43]:
confusion_matrix(y_true=lab,
                 y_pred=predict)

array([[ 5, 22, 77,  0,  9,  0,  9, 13,  0,  0],
       [ 0, 11, 31,  0,  9,  0,  3,  3,  0,  0],
       [ 4, 21, 71,  0, 13,  0,  1, 20,  0,  0],
       [ 2, 14, 17,  0, 13,  0,  2, 12,  0,  0],
       [ 4, 15, 60,  0, 14,  0,  8, 14,  0,  0],
       [ 0, 15, 28,  0, 14,  0, 10, 27,  0,  0],
       [ 5, 18, 50,  0,  8,  0,  8, 12,  0,  0],
       [ 1, 37, 86,  0, 25,  0, 13, 41,  0,  0],
       [ 2, 15, 46,  0, 15,  0,  9, 18,  0,  0],
       [ 0,  5,  3,  0,  2,  0,  0, 14,  0,  0]])